# 06 — Feature Selection & Diagnostics

**Objective:** Evaluate feature redundancy, Variance Inflation Factor (VIF), and mutual information scores across both prediction tracks.


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.feature_selection import mutual_info_regression

def find_root():
    p = Path.cwd()
    for parent in [p] + list(p.parents):
        if (parent / 'data' / 'raw' / 'gt_2011.csv').exists():
            return parent
    return p

ROOT = find_root()
sys.path.insert(0, str(ROOT))

from src.data.loader import FEATURES_A, TARGETS_A


## 1. Mutual Information Scoring with Emission Targets


In [2]:
df = pd.read_parquet(ROOT / "data" / "processed" / "cleaned_master.parquet")
# Subsample for fast MI computation
sample = df.sample(n=5000, random_state=42)
X_sub = sample[FEATURES_A]

mi_co = mutual_info_regression(X_sub, sample["CO"], random_state=42)
mi_nox = mutual_info_regression(X_sub, sample["NOX"], random_state=42)

diag_df = pd.DataFrame({
    "feature": FEATURES_A,
    "mutual_info_CO": np.round(mi_co, 4),
    "mutual_info_NOX": np.round(mi_nox, 4)
}).sort_values("mutual_info_CO", ascending=False)

print(diag_df.to_string(index=False))

out_diag = ROOT / "outputs" / "tables" / "feature_diagnostics.csv"
diag_df.to_csv(out_diag, index=False)
print(f"Saved feature diagnostics to {out_diag.relative_to(ROOT)}")
print("Feature selection and diagnostics COMPLETE.")


feature  mutual_info_CO  mutual_info_NOX
    TIT          0.5326           0.2410
    TEY          0.4812           0.2508
    CDP          0.4671           0.2231
   GTEP          0.4535           0.2633
   AFDP          0.2894           0.1379
    TAT          0.1606           0.1134
     AT          0.1019           0.2625
     AP          0.0584           0.1000
     AH          0.0304           0.0368
Saved feature diagnostics to outputs\tables\feature_diagnostics.csv
Feature selection and diagnostics COMPLETE.
